# 행정경계 시계열 로더

### 폴더 구조
```
GTFS_CT/
└── 행정경계/
    ├── 2016/  EA001G.shp  (시도/시군구/읍면동 통합)
    ├── 2017/
    └── ...2024/
```

### 테이블: EA001G (행정경계)
| 필드명(Full) | shp 필드명 | 내용 | 타입 |
|---|---|---|---|
| DISTRICT_ID | DISTRICT_I | 행정구역 ID | VARCHAR2(8) |
| DISTRICT_NAME | DISTRICT_N | 행정구역명 | VARCHAR2(30) |
| DISTRICT_TYPE | DISTRICT_T | 유형 (2=시도, 3=시군구, 4=읍면동) | VARCHAR2(1) |
| X_COORDINATE | X | X좌표 (중심) | Double |
| Y_COORDINATE | Y | Y좌표 (중심) | Double |
| UPDDISTRICT_ID | UPDDISTRICT | 상위행정구역 ID | VARCHAR2(7) |
| AREA | AREA | 면적 | Double |

In [14]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

In [15]:
BASE_DIR  = Path(r"C:\Users\HP\Desktop\작업 폴더\01_work\아산시\01_데이터\원천데이터\교통약자_이동권_분석\GTFS\GTFS_CT")
ADMIN_DIR = BASE_DIR / "행정경계"

# 아산시 충남 시군구 코드 (DISTRICT_ID 앞 5자리)
ASAN_CODE      = "44200"    # 아산시
CHUNGNAM_CODE  = "44"       # 충청남도

# WGS84 bbox (아산시)
ASAN_BOUNDS_WGS84 = (126.8, 36.6, 127.1, 36.9)

In [16]:
# shp 필드명(10자 잘림) → 풀네임 정규화
RENAME = {
    "DISTRICT_I": "DISTRICT_ID",
    "DISTRICT_N": "DISTRICT_NAME",
    "DISTRICT_T": "DISTRICT_TYPE",
    "X_COORDINA": "X_COORDINATE",
    "Y_COORDINA": "Y_COORDINATE",
    "UPDDISTRICT": "UPDDISTRICT_ID",  # 11자라 그대로일 수 있음
    "UPDISTRICT": "UPDDISTRICT_ID",
}

# DISTRICT_TYPE 코드 레이블
TYPE_LABEL = {"2": "시도", "3": "시군구", "4": "읍면동"}

In [17]:
def find_shp(folder: Path, keyword: str = "") -> Path | None:
    for shp in folder.rglob("*.shp"):
        if not keyword or keyword.lower() in shp.stem.lower():
            return shp
    return None


def load_admin(shp_path: Path, year: int) -> gpd.GeoDataFrame | None:
    if shp_path is None or not shp_path.exists():
        return None
    try:
        gdf = gpd.read_file(shp_path, engine="pyogrio")
        gdf.columns = [c.upper() for c in gdf.columns]
        gdf = gdf.rename(columns={k.upper(): v.upper() for k, v in RENAME.items()})
        gdf["data_year"] = year
        return gdf
    except Exception as e:
        print(f"  [ERR] {shp_path.name} ({year}): {e}")
        return None


def filter_bbox(gdf: gpd.GeoDataFrame, bounds: tuple) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        print("  [WARN] CRS 없음 - EPSG:5179 가정")
        gdf = gdf.set_crs(epsg=5179)
    wgs = gdf.to_crs(epsg=4326)
    minx, miny, maxx, maxy = bounds
    return wgs.cx[minx:maxx, miny:maxy].copy()

In [18]:
# 폴더 구조 확인
print(f"[행정경계] {ADMIN_DIR}")
for year_dir in sorted(ADMIN_DIR.iterdir()):
    if not year_dir.is_dir():
        continue
    shps = list(year_dir.rglob("*.shp"))
    print(f"  {year_dir.name}/  shp: {[s.name for s in shps]}")

[행정경계] C:\Users\HP\Desktop\작업 폴더\01_work\아산시\01_데이터\원천데이터\교통약자_이동권_분석\GTFS\GTFS_CT\행정경계
  2016/  shp: ['EA001G.shp']
  2017/  shp: ['EA001G_2016기준.shp']
  2018/  shp: ['EA001G_2017.shp']
  2019/  shp: ['EA001G_2018.shp']
  2020/  shp: ['EA001G_2019기준_세계.shp']
  2021/  shp: ['EA001G_2020.shp']
  2022/  shp: ['EA001G_2021_GR.shp']
  2023/  shp: ['EA001G_2022_GR.shp']
  2024/  shp: ['EA001G_2023_GR.shp']


---
## 1. 전체 로드

In [19]:
all_years = []

for year_dir in sorted(ADMIN_DIR.iterdir()):
    if not year_dir.is_dir() or not year_dir.name.isdigit():
        continue
    year = int(year_dir.name)

    # EA001G 우선, 없으면 첫 번째 shp
    shp = find_shp(year_dir, "ea001") or find_shp(year_dir, "admin") or find_shp(year_dir)
    gdf = load_admin(shp, year)

    if gdf is not None:
        # GEOMETRY 컬럼을 geometry로 설정
        if 'GEOMETRY' in gdf.columns:
            gdf = gdf.set_geometry('GEOMETRY')

        # CRS 통일 (EPSG:5179 - 한국 표준 좌표계)
        if gdf.crs is not None and gdf.crs.to_epsg() != 5179:
            gdf = gdf.to_crs(epsg=5179)
        elif gdf.crs is None:
            print(f"  [WARN] {year} CRS 없음 - EPSG:5179 가정")
            gdf = gdf.set_crs(epsg=5179)

        all_years.append(gdf)
        n_sido = (gdf["DISTRICT_TYPE"] == "2").sum() if "DISTRICT_TYPE" in gdf.columns else "?"
        n_sigg = (gdf["DISTRICT_TYPE"] == "3").sum() if "DISTRICT_TYPE" in gdf.columns else "?"
        n_emd = (gdf["DISTRICT_TYPE"] == "4").sum() if "DISTRICT_TYPE" in gdf.columns else "?"
        print(f"  [OK] {year}  전체 {len(gdf):,}행  시도:{n_sido}  시군구:{n_sigg}  읍면동:{n_emd}  crs=EPSG:5179")
    else:
        print(f"  [--] {year}  shp 없음")

admin_all = pd.concat(all_years, ignore_index=True) if all_years else None
print(f"\n전체 로드: {len(admin_all):,}행" if admin_all is not None else "데이터 없음")


  [OK] 2016  전체 3,771행  시도:17  시군구:252  읍면동:3502  crs=EPSG:5179
  [OK] 2017  전체 3,770행  시도:17  시군구:250  읍면동:3503  crs=EPSG:5179
  [OK] 2018  전체 3,767행  시도:17  시군구:250  읍면동:3500  crs=EPSG:5179
  [OK] 2019  전체 3,771행  시도:17  시군구:250  읍면동:3504  crs=EPSG:5179
  [OK] 2020  전체 3,758행  시도:0  시군구:0  읍면동:0  crs=EPSG:5179
  [OK] 2021  전체 3,768행  시도:0  시군구:0  읍면동:0  crs=EPSG:5179
  [OK] 2022  전체 3,779행  시도:17  시군구:250  읍면동:3512  crs=EPSG:5179
  [OK] 2023  전체 3,788행  시도:17  시군구:250  읍면동:3521  crs=EPSG:5179
  [OK] 2024  전체 3,799행  시도:17  시군구:252  읍면동:3530  crs=EPSG:5179

전체 로드: 33,971행


---
## 2. 아산시 필터링

두 가지 방식 중 선택:
- **A) bbox 클립** — geometry 기반, 경계에 걸친 폴리곤도 포함
- **B) DISTRICT_ID 코드 기반** — 정확한 행정구역 단위 추출 (권장)

In [24]:
# 아산시 시군구 코드 (예: 44200)
ASAN_SIGUNGU_CODE = "44200"  # 실제 코드로 수정 필요

# 아산시 전체 (시군구 + 하위 읍면동)
admin_asan = admin_all[
    admin_all["DISTRICT_ID"].str.startswith(ASAN_SIGUNGU_CODE[:5], na=False)
].copy()

print(f"아산시 행정경계: {len(admin_asan)}행")
print(admin_asan.head())

아산시 행정경계: 0행
Empty GeoDataFrame
Columns: [OBJECTID, DISTRICT_ID, DISTRICT_NAME, DISTRICT_TYPE, X_COORDINATE, Y_COORDINATE, UPDDISTRICT_ID, AREA, GEOMETRY, data_year, X, Y]
Index: []


In [20]:
if admin_all is None:
    print("데이터 없음")
else:
    # 방식 B: DISTRICT_ID 기반 필터 (더 정확)
    # 아산시(44200) 자체 + 하위 읍면동 (4420*)
    asan_mask = admin_all["DISTRICT_ID"].str.startswith(ASAN_CODE) | \
                (admin_all["DISTRICT_ID"] == ASAN_CODE)
    asan_admin = admin_all[asan_mask].copy()

    print(f"아산시 행정경계: {len(asan_admin):,}행")
    if "DISTRICT_TYPE" in asan_admin.columns:
        print(asan_admin.groupby(["data_year", "DISTRICT_TYPE"])["DISTRICT_ID"].count().unstack(fill_value=0).rename(columns=TYPE_LABEL).to_string())

아산시 행정경계: 0행
Empty DataFrame
Columns: []
Index: []


In [21]:
# 아산시 읍면동 목록 확인 (최신 연도 기준)
if admin_all is not None and "DISTRICT_TYPE" in admin_all.columns:
    latest = admin_all["data_year"].max()
    emd = admin_all[
        (admin_all["data_year"] == latest) &
        (admin_all["DISTRICT_ID"].str.startswith(ASAN_CODE)) &
        (admin_all["DISTRICT_TYPE"] == "4")
    ][["DISTRICT_ID", "DISTRICT_NAME"]].sort_values("DISTRICT_ID")
    print(f"[{latest}년 기준] 아산시 읍면동 {len(emd)}개")
    print(emd.to_string(index=False))

[2024년 기준] 아산시 읍면동 0개
Empty DataFrame
Columns: [DISTRICT_ID, DISTRICT_NAME]
Index: []


In [22]:
# 연도별 행정구역 변동 확인 (읍면동 수 변화)
if admin_all is not None and "DISTRICT_TYPE" in admin_all.columns:
    changes = (
        admin_all[
            admin_all["DISTRICT_ID"].str.startswith(ASAN_CODE) &
            (admin_all["DISTRICT_TYPE"] == "4")
        ]
        .groupby("data_year")["DISTRICT_ID"]
        .count()
        .rename("읍면동수")
    )
    print("아산시 읍면동 수 연도별 변화:")
    print(changes.to_string())

아산시 읍면동 수 연도별 변화:
Series([], )


---
## 3. CSV 저장

In [23]:
def save_csv(df, path: Path):
    if hasattr(df, "geometry") and "geometry" in df.columns:
        df = df.drop(columns=["geometry"])
    try:
        df.to_csv(path, index=False, encoding="cp949")
    except UnicodeEncodeError:
        path = path.with_stem(path.stem + "_utf8")
        df.to_csv(path, index=False, encoding="utf-8-sig")
        print(f"  [fallback utf8] {path.name}")
    print(f"  [저장] {path.name}  {df.shape}")


SAVE_DIR = ADMIN_DIR / "_csv"
SAVE_DIR.mkdir(exist_ok=True)

# 아산시만
if asan_admin is not None:
    save_csv(asan_admin, SAVE_DIR / "admin_asan.csv")

# 전국 (용량 주의)
save_all = input("전국 데이터도 저장? (y/n): ").strip().lower() == "y"
if save_all and admin_all is not None:
    save_csv(admin_all, SAVE_DIR / "admin_all.csv")

print(f"\n저장 경로: {SAVE_DIR}")

  [저장] admin_asan.csv  (0, 12)
  [저장] admin_all.csv  (33971, 12)

저장 경로: C:\Users\HP\Desktop\작업 폴더\01_work\아산시\01_데이터\원천데이터\교통약자_이동권_분석\GTFS\GTFS_CT\행정경계\_csv
